In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!gdown --id 1lmz_vfsXJi7rMvLZ9jMAEQKa_iV1tH5w

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1lmz_vfsXJi7rMvLZ9jMAEQKa_iV1tH5w
From (redirected): https://drive.google.com/uc?id=1lmz_vfsXJi7rMvLZ9jMAEQKa_iV1tH5w&confirm=t&uuid=e0a097c5-d61e-4b0e-a24d-bcdf0a71aafb
To: /kaggle/working/updated_OCR_Text.zip
100%|█████████████████████████████████████████| 285M/285M [00:02<00:00, 124MB/s]


In [3]:
!unzip updated_OCR_Text.zip

Archive:  updated_OCR_Text.zip
  inflating: 2316.jpg                
  inflating: 2807.jpg                
  inflating: 15604.jpg               
  inflating: 14008.jpg               
  inflating: 13765.txt               
  inflating: 14323.jpg               
  inflating: 726.txt                 
  inflating: 8101.jpg                
  inflating: 15584.jpg               
  inflating: 8718.jpg                
  inflating: 11423.jpg               
  inflating: 16904.jpg               
  inflating: 19960.txt               
  inflating: 2420.txt                
  inflating: 19421.txt               
  inflating: 4597.txt                
  inflating: 17940.jpg               
  inflating: 16683.txt               
  inflating: 5906.jpg                
  inflating: 2212.txt                
  inflating: 737.jpg                 
  inflating: 18936.jpg               
  inflating: 1502.jpg                
  inflating: 6783.txt                
  inflating: 17107.jpg               
  inflating: 1742.j

In [4]:
import os
import glob

# الكود ده هيبحث عن أي صورة jpg جوه كاجل عشان يحدد المسار لوحده
all_images = glob.glob("/kaggle/working/**/*.jpg", recursive=True)

if len(all_images) == 0:
    print("لم يتم العثور على أي صور! تأكد إن خلية الـ unzip خلصت شغل ومطلعتش Error.")
else:
    # استنتاج المسار من أول صورة لقاها
    dataset_path = os.path.dirname(all_images[0])
    print(f"تم تحديد مسار الداتا بنجاح: {dataset_path}")

    image_paths = []
    labels = []

    # قراءة كل الصور من المسار اللي تم اكتشافه
    image_files = glob.glob(os.path.join(dataset_path, "*.jpg"))

    for img_path in image_files:
        txt_path = img_path.replace(".jpg", ".txt")
        if os.path.exists(txt_path):
            with open(txt_path, 'r', encoding='utf-8') as f:
                text = f.read().strip()
                if text:
                    image_paths.append(img_path)
                    labels.append(text)

    print(f"Total valid samples found: {len(image_paths)}")

    def get_vocabulary(labels):
        characters = set(char for label in labels for char in label)
        vocab_chars = sorted(list(characters))
        print(f"Total unique characters: {len(vocab_chars)}")
        return vocab_chars

    vocab_chars = get_vocabulary(labels)

تم تحديد مسار الداتا بنجاح: /kaggle/working
Total valid samples found: 19992
Total unique characters: 64


In [5]:
import os
import glob
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ==========================================
# 1. قراءة الداتا وتجهيز القاموس
# ==========================================
all_images = glob.glob("/kaggle/working/**/*.jpg", recursive=True)

image_paths = []
labels = []

for img_path in all_images:
    txt_path = img_path.replace(".jpg", ".txt")
    if os.path.exists(txt_path):
        with open(txt_path, 'r', encoding='utf-8') as f:
            text = f.read().strip()
            if text:
                image_paths.append(img_path)
                labels.append(text)

vocab_chars = sorted(list(set(char for label in labels for char in label)))
char_to_num = tf.keras.layers.StringLookup(vocabulary=vocab_chars, mask_token=None)

# ==========================================
# 2. تجهيز البايب لاين للصور والنصوص
# ==========================================
img_width = 600 # العرض الجديد عشان يستوعب العناوين الطويلة
img_height = 50
batch_size = 32

def encode_single_sample(img_path, label):
    img = tf.io.read_file(img_path)
    img = tf.io.decode_jpeg(img, channels=1) 
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, [img_height, img_width])
    img = tf.transpose(img, perm=[1, 0, 2])
    label = char_to_num(tf.strings.unicode_split(label, input_encoding="UTF-8"))
    return {"image": img, "label": label}

dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
dataset = dataset.map(encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.padded_batch(batch_size, padded_shapes={"image": [img_width, img_height, 1], "label": [None]})
dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

train_size = int(0.9 * (len(image_paths) // batch_size))
train_dataset = dataset.take(train_size)
validation_dataset = dataset.skip(train_size)

# ==========================================
# 3. بناء الموديل (مع التعديل الجذري للـ CTC)
# ==========================================
@tf.keras.utils.register_keras_serializable(package="Custom", name="CTCLayer")
class CTCLayer(layers.Layer):
    def __init__(self, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.loss_fn = keras.backend.ctc_batch_cost

    def call(self, y_true, y_pred):
        batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
        input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
        
        # 🔥 الحل السحري: حساب عدد الحروف الحقيقية وتجاهل أصفار الـ Padding
        label_length = tf.math.count_nonzero(y_true, axis=-1, keepdims=True, dtype="int64")

        input_length = input_length * tf.ones(shape=(batch_len, 1), dtype="int64")

        loss = self.loss_fn(y_true, y_pred, input_length, label_length)
        self.add_loss(loss)
        return y_pred

    def get_config(self):
        return super().get_config()

def build_model():
    input_img = layers.Input(shape=(img_width, img_height, 1), name="image", dtype="float32")
    labels = layers.Input(name="label", shape=(None,), dtype="float32")

    x = layers.Conv2D(32, (3, 3), activation="relu", kernel_initializer="he_normal", padding="same", name="Conv1")(input_img)
    x = layers.MaxPooling2D((2, 2), name="pool1")(x)

    x = layers.Conv2D(64, (3, 3), activation="relu", kernel_initializer="he_normal", padding="same", name="Conv2")(x)
    x = layers.MaxPooling2D((2, 2), name="pool2")(x)

    new_shape = ((img_width // 4), (img_height // 4) * 64)
    x = layers.Reshape(target_shape=new_shape, name="reshape")(x)
    x = layers.Dense(64, activation="relu", name="dense1")(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.25))(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True, dropout=0.25))(x)

    num_classes = len(char_to_num.get_vocabulary()) + 1
    x = layers.Dense(num_classes, activation="softmax", name="dense2")(x)

    output = CTCLayer(name="ctc_loss")(labels, x)

    model = keras.models.Model(inputs=[input_img, labels], outputs=output, name="ocr_model_v1")
    prediction_model = keras.models.Model(inputs=input_img, outputs=x, name="prediction_model")
    
    return model, prediction_model

model, prediction_model = build_model()
model.compile(optimizer=keras.optimizers.Adam())

# ==========================================
# 4. بدء التدريب
# ==========================================
print("🚀 Starting training with FIXED CTC Lengths...")
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=40, 
)

model_save_path = "best_ocr_model.keras"
prediction_model.save(model_save_path)
print(f"Model saved successfully to {model_save_path}")

I0000 00:00:1785847706.284606      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785847706.287315      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


🚀 Starting training with FIXED CTC Lengths...
Epoch 1/40
561/561 ━━━━━━━━━━━━━━━━━━━━ 56s 86ms/step - loss: 4328.7993 - val_loss: 3101.6011
Epoch 2/40
561/561 ━━━━━━━━━━━━━━━━━━━━ 47s 84ms/step - loss: 2937.7827 - val_loss: 2817.9326
Epoch 3/40
561/561 ━━━━━━━━━━━━━━━━━━━━ 50s 89ms/step - loss: 2834.3970 - val_loss: 2768.6313
Epoch 4/40
561/561 ━━━━━━━━━━━━━━━━━━━━ 47s 84ms/step - loss: 2796.9668 - val_loss: 2748.5864
Epoch 5/40
561/561 ━━━━━━━━━━━━━━━━━━━━ 47s 84ms/step - loss: 2766.9531 - val_loss: 2727.7905
Epoch 6/40
561/561 ━━━━━━━━━━━━━━━━━━━━ 47s 83ms/step - loss: 2798.5447 - val_loss: 2719.7209
Epoch 7/40
561/561 ━━━━━━━━━━━━━━━━━━━━ 50s 89ms/step - loss: 2756.8003 - val_loss: 2697.0032
Epoch 8/40
561/561 ━━━━━━━━━━━━━━━━━━━━ 47s 84ms/step - loss: 2720.1987 - val_loss: 2684.1851
Epoch 9/40
561/561 ━━━━━━━━━━━━━━━━━━━━ 50s 89ms/step - loss: 2728.5271 - val_loss: 2683.0295
Epoch 10/40
561/561 ━━━━━━━━━━━━━━━━━━━━ 47s 83ms/step - loss: 2707.6079 - val_loss: 2688.3264
Epoch 11/40
5

In [6]:
import json
from IPython.display import FileLink, display

# 1. حفظ قاموس الحروف في ملف json (عشان نستخدمه في الـ Backend)
with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_chars, f, ensure_ascii=False)

print("🎉 جاهز للتحميل! اضغط على الروابط دي عشان تنزل الملفات على جهازك:")

# 2. عمل روابط تحميل مباشرة
display(FileLink("best_ocr_model.keras"))
display(FileLink("vocab.json"))

🎉 جاهز للتحميل! اضغط على الروابط دي عشان تنزل الملفات على جهازك:


/kaggle/working/best_ocr_model.keras

/kaggle/working/vocab.json